---

Hotel Booking ... 

---

### **1.Introduction**

# =============================================================================
# HOTEL BOOKING CANCELLATION PREDICTION
# Final Assignment – Data Science Course
# Dataset : hotel_bookings.csv  (119,390 bookings, 32 columns)
# Target  : is_canceled  (0 = kept, 1 = cancelled)
# =============================================================================

# ---------------------------------------------------------------------------
# CELL 1 – IMPORTS
# ---------------------------------------------------------------------------
# We load every library up-front so the rest of the notebook runs cleanly.
# Key libraries:
#   pandas / numpy          → data manipulation and maths
#   matplotlib / seaborn    → plots
#   scipy.stats             → hypothesis tests
#   sklearn                 → machine learning pipeline, models, metrics
#   optuna                  → automatic hyperparameter tuning

### **2. Setup**

#### 2.1. Library

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "optuna", "-q"],
               capture_output=True)

import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss, brier_score_loss,
)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
sns.set_theme(context="talk", style="whitegrid", font_scale=0.85)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

print("✅ All libraries loaded.")



✅ All libraries loaded.


c:\Users\Admin\Documents\ML\group1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 2.2. Load data

In [2]:
DATA_PATH = r"..\data\raw\raw_data.csv"

print(f"Loading: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH, dtype={"agent": "Float64", "company": "Float64"})

print(f"Shape : {df_raw.shape}   ({df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns)")
print(f"\nTarget (is_canceled) distribution:")
vc = df_raw["is_canceled"].value_counts()
print(f"  Not cancelled (0) : {vc[0]:,}  ({vc[0]/len(df_raw):.1%})")
print(f"  Cancelled     (1) : {vc[1]:,}  ({vc[1]/len(df_raw):.1%})")



Loading: ..\data\raw\raw_data.csv
Shape : (119390, 32)   (119,390 rows, 32 columns)

Target (is_canceled) distribution:
  Not cancelled (0) : 75,166  (63.0%)
  Cancelled     (1) : 44,224  (37.0%)


### 3. Data Audit

Drop các data leakage, in order to prevent ...    
- Feature name/ Desctip/ Reasons
...

# ---------------------------------------------------------------------------
# CELL 3 – DATA AUDIT  (types & missing values)
# ---------------------------------------------------------------------------
# Before touching anything, we check:
#   pct_null   – columns with high missingness need special handling
#   n_unique   – very high cardinality (e.g. 177 countries) is expensive to encode
#   dtype      – confirm numeric columns are not stored as strings


In [3]:
print("=== dtype / missing-value audit ===")
audit = pd.DataFrame({
    "dtype"    : df_raw.dtypes,
    "n_null"   : df_raw.isna().sum(),
    "pct_null" : (df_raw.isna().mean() * 100).round(1),
    "n_unique" : df_raw.nunique(),
})
print(audit.sort_values("pct_null", ascending=False).to_string())



=== dtype / missing-value audit ===
                                  dtype  n_null  pct_null  n_unique
company                         Float64  112593      94.3       352
agent                           Float64   16340      13.7       333
country                          object     488       0.4       177
hotel                            object       0       0.0         2
previous_cancellations            int64       0       0.0        15
reservation_status               object       0       0.0         3
total_of_special_requests         int64       0       0.0         6
required_car_parking_spaces       int64       0       0.0         5
adr                             float64       0       0.0      8879
customer_type                    object       0       0.0         4
days_in_waiting_list              int64       0       0.0       128
deposit_type                     object       0       0.0         3
booking_changes                   int64       0       0.0        21
assigned_roo

In [4]:

# ---------------------------------------------------------------------------
# CELL 4 – LEAKAGE AUDIT
# ---------------------------------------------------------------------------
# Data leakage = using information that only exists AFTER the outcome is known.
# A model trained with leaky features fails the moment it faces real data —
# it would need to know the future to make a prediction.
#
# reservation_status      → literally says "Canceled / Check-Out / No-Show"
# reservation_status_date → the date the cancellation was recorded
# Both are dropped immediately.

print("=== Leakage & exclusion decisions ===\n")
decisions = [
    ("reservation_status",      "❌ LEAKAGE",    "Directly encodes the outcome."),
    ("reservation_status_date", "❌ LEAKAGE",    "Only exists for cancelled rows."),
    ("country",                 "⚠️  EXCLUDED",  "177 categories + partial leakage risk."),
    ("agent",                   "⚠️  EXCLUDED",  "333 unique IDs; high cardinality."),
    ("company",                 "⚠️  EXCLUDED",  "94 % missing."),
    ("arrival_date_*",          "📅 KEPT (meta)","Not a feature; used only to split by time."),
]
for col, dec, reason in decisions:
    print(f"  {dec:20s}  {col:30s}  {reason}")



=== Leakage & exclusion decisions ===

  ❌ LEAKAGE             reservation_status              Directly encodes the outcome.
  ❌ LEAKAGE             reservation_status_date         Only exists for cancelled rows.
  ⚠️  EXCLUDED          country                         177 categories + partial leakage risk.
  ⚠️  EXCLUDED          agent                           333 unique IDs; high cardinality.
  ⚠️  EXCLUDED          company                         94 % missing.
  📅 KEPT (meta)         arrival_date_*                  Not a feature; used only to split by time.


In [5]:

# ---------------------------------------------------------------------------
# CELL 5 – SELECT SOURCE COLUMNS
# ---------------------------------------------------------------------------
# We bring in only the raw columns that feed into our final feature table.
# Three extra date columns are added purely so we can reconstruct arrival_date
# later for the time-based split — they are dropped before model training.

TARGET = "is_canceled"

DATE_COLS   = ["arrival_date_year", "arrival_date_month", "arrival_date_day_of_month"]
SOURCE_COLS = [
    # raw numerics → used directly or as ingredients for engineered features
    "lead_time", "required_car_parking_spaces", "total_of_special_requests",
    "previous_cancellations", "previous_bookings_not_canceled",
    "adr", "adults", "children", "babies",
    # categoricals → used directly after one-hot encoding
    "deposit_type", "market_segment", "distribution_channel", "customer_type", "hotel",
] + DATE_COLS

df = df_raw[SOURCE_COLS + [TARGET]].copy()
df["children"] = df["children"].fillna(0).astype(int)  # fill 4 missing children values

print(f"Working frame: {df.shape}")
print(f"Columns: {df.columns.tolist()}")



Working frame: (119390, 18)
Columns: ['lead_time', 'required_car_parking_spaces', 'total_of_special_requests', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'adults', 'children', 'babies', 'deposit_type', 'market_segment', 'distribution_channel', 'customer_type', 'hotel', 'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month', 'is_canceled']


### 5. Save data

In [7]:
df.to_csv("..\data\cleaned\cleaned.csv", index=False)